**Nested cross-validation**, a powerful technique used for robust model evaluation, especially when hyperparameter tuning is involved.

It's common to use nested cross-validation to avoid optimistic bias in performance estimates when hyperparameter tuning is performed on the same data used for evaluation.

**Problem without Nesting**: If you perform hyperparameter tuning (like GridSearchCV) on your entire dataset and then evaluate the best model on the same dataset (or a simple train-test split after tuning), you introduce optimization bias (also called data leakage or peeking). The model has "seen" the test data during tuning, so its reported performance will be optimistically inflated and won't reflect how it performs on truly unseen data.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import(
    train_test_split,
    cross_validate,
    GridSearchCV,
    KFold
)

In [3]:
x,y = load_breast_cancer(return_X_y = True,as_frame = True)
y = y.map({1:0,0:1})
x.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.3,random_state = 0)
x_train.shape,x_test.shape

((398, 30), (171, 30))

In [ ]:
#reset index to avoid issues with indices after splitting
x_train.reset_index(drop = True,inplace = True)
y_train.reset_index(drop = True,inplace = True)

This function takes two main arguments:

    1. 'model': An unfitted machine learning model (e.g., LogisticRegression(), SVC()).
       This is the base model that will be trained and evaluated.
       
    2. 'grid': A dictionary defining the hyperparameter search space for GridSearchCV.
       e.g., {'C': [0.1, 1.0, 10.0], 'kernel': ['linear', 'rbf']}

**Solution**: Nesting: You create two "loops" of cross-validation:

**Outer Loop**: Divides the data into completely separate "training" and "test" sets. The "test" set in this outer loop is held back and never touched during hyperparameter tuning.

**Inner Loop:** Performs the hyperparameter tuning (e.g., GridSearchCV) only on the "training" data from the current outer loop fold. This inner loop also uses cross-validation to find the best hyperparameters.

**Evaluation**: The model with the best hyperparameters (found by the inner loop) is then evaluated on the truly unseen "test" set from the outer loop.

This process is repeated for each fold of the outer loop, and the results

In [ ]:
def nested_cross_val(model, grid):
    cv_outer = KFold(n_splits = 5, shuffle = True, random_state = 0) #for unbiased performance estimate
    cv_inner = KFold(n_splits = 5, shuffle = True, random_state = 0) #for hyper parameter tuning
    
    #for storing results
    outer_results = list()
    immer_results = list()
    
    #let's iterate through the outer fold
    for train_ix, test_ix in cv_outer.split(x_train):
        

If you simply ran GridSearchCV once on your entire x_train, GridSearchCV would report a best_score. However, this score would be an optimistically biased estimate of your model's true performance because the validation folds used by GridSearchCV were still part of the data used for hyperparameter selection. Nested cross-validation ensures that the final performance estimate (outer_results) is obtained on data that was completely unseen during the hyperparameter tuning phase of each fold, providing a more realistic and less biased measure of your model's generalization ability.

In [8]:
logit = LogisticRegression(penalty = 'l2',C=1,solver = 'liblinear',max_iter=1000, random_state=4)
logit_grid_param  = dict(
    C = [0.1,0.5,1.0,5.0,10.0],
    penalty = ['l1', 'l2']  # l1 is not supported by 'liblinear' solver, but included for completeness.
)
logit_search = nested_cross_val(logit,logit_grid_param)
logit_search

>accuracy_outer=0.975,accuracy_inner=0.953,cfg={'C': 10.0, 'penalty': 'l1'}
>accuracy_outer=0.963,accuracy_inner=0.943,cfg={'C': 5.0, 'penalty': 'l1'}
>accuracy_outer=0.925,accuracy_inner=0.959,cfg={'C': 10.0, 'penalty': 'l1'}
>accuracy_outer=0.924,accuracy_inner=0.956,cfg={'C': 10.0, 'penalty': 'l1'}
>accuracy_outer=0.962,accuracy_inner=0.950,cfg={'C': 10.0, 'penalty': 'l1'}


([0.975, 0.9625, 0.925, 0.9240506329113924, 0.9620253164556962],
 [np.float64(0.9528769841269842),
  np.float64(0.9433035714285714),
  np.float64(0.959077380952381),
  np.float64(0.9561507936507937),
  np.float64(0.9498511904761905)])

In [10]:
x_train_preds = logit_search[2].predict(x_train)
x_test_preds = logit_search[2].predict(x_test)

IndexError: tuple index out of range